# Fine-tune Universal Formula NN (frozen backbone)

Adapts the pretrained model to the single-integrator CLF+CBF problem.

**Files needed in Kaggle dataset:**
- `best_kf_model_N10_m10.pt`
- `scaler_X_N10_m10.pkl`
- `scaler_y_N10_m10.pkl`


## Imports & device

In [ ]:
import os, time, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import r2_score
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
import multiprocessing as mp

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cpu')
WORK_DIR = '.'

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'Device  : {DEVICE}')

## Config

In [ ]:
# Problem dimensions
N_MAX  = 10
M_DIM  = 10
IN_DIM = N_MAX * (1 + M_DIM) + 1   # 111

# Data generation 
N_FINETUNE   = 40_000

X_RADIUS_MIN = 0.3
X_RADIUS_MAX = 5.0
OBS_DIST_MIN = 1.0
OBS_DIST_MAX = 5.0
OBS_R_MIN    = 0.3
OBS_R_MAX    = 1.5
C_CLF_MIN    = 0.05
C_CLF_MAX    = 2.0
CBF_GAIN     = 8.0 

# Training 
BATCH_SIZE   = 256
MAX_EPOCHS   = 150
PATIENCE     = 20
LR_FINETUNE  = 5e-5
WEIGHT_DECAY = 1e-4
VAL_FRAC     = 0.10

# Gradient-flow ODE
ODE_RTOL = 1e-6
ODE_ATOL = 1e-6
ODE_TMAX = 50.0
GRAD_TOL = 1e-6

print(f'IN_DIM={IN_DIM}  N_FINETUNE={N_FINETUNE}')
print(f'n_cbf: 1..{N_MAX-1}  (uniform)    n_dim: n_cbf..{M_DIM}')
print(f'Expected time: ~{N_FINETUNE/40/60:.0f} min data gen + ~5 min training')

## Load pretrained model and scalers

In [ ]:
INPUT_DIR = '.'

def find_file(name):
    # First try walking /kaggle/input for the basename
    basename = os.path.basename(name)
    for root, dirs, files in os.walk(INPUT_DIR):
        if basename in files:
            return os.path.join(root, basename)
    # Fallback: exact path
    if os.path.exists(name):
        return name
    raise FileNotFoundError(f'{basename} not found under {INPUT_DIR}')

MODEL_PATH = find_file('best_kf_model_N10_m10.pt')
SCALER_X   = find_file('scaler_X_N10_m10.pkl')
SCALER_Y   = find_file('scaler_y_N10_m10.pkl')
print('Model   :', MODEL_PATH)
print('Scaler X:', SCALER_X)
print('Scaler y:', SCALER_Y)

scaler_X = joblib.load(SCALER_X)
scaler_y = joblib.load(SCALER_Y)
print(f'scaler_y scale: min={scaler_y.scale_.min():.3f}  max={scaler_y.scale_.max():.3f}')

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim), nn.SiLU(),
            nn.Linear(dim, dim), nn.Dropout(dropout),
            nn.LayerNorm(dim), nn.SiLU(),
            nn.Linear(dim, dim), nn.Dropout(dropout),
        )
    def forward(self, x): return x + self.block(x)

class KfNet(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=256, n_blocks=5, dropout=0.15):
        super().__init__()
        self.stem   = nn.Linear(in_dim, hidden)
        self.blocks = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head   = nn.Sequential(nn.LayerNorm(hidden), nn.SiLU(), nn.Linear(hidden, out_dim))
    def forward(self, x): return self.head(self.blocks(self.stem(x)))


# ── Fine-tune wrapper: frozen backbone + trainable adapter head ──────────
class KfNetFineTune(nn.Module):
    """
    Wraps a pretrained KfNet.
    - backbone (stem + ResBlocks): ALL weights FROZEN.
    - ft_head: small MLP added on top, trained from scratch on new data.
    The frozen backbone produces a 'hidden'-dim feature vector.
    The ft_head maps that to out_dim.
    """
    def __init__(self, pretrained: KfNet, hidden=256, out_dim=10,
                 adapter_dim=128, dropout=0.15):
        super().__init__()
        # Frozen backbone: stem + resblocks
        self.backbone = nn.Sequential(pretrained.stem, pretrained.blocks)
        for p in self.backbone.parameters():
            p.requires_grad = False

        # Trainable fine-tune head
        self.ft_head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.SiLU(),
            nn.Linear(hidden, adapter_dim),
            nn.Dropout(dropout),
            nn.LayerNorm(adapter_dim),
            nn.SiLU(),
            nn.Linear(adapter_dim, out_dim),
        )

    def forward(self, x):
        with torch.no_grad():           # no gradients through frozen backbone
            features = self.backbone(x)
        return self.ft_head(features)


# Load pretrained weights into base KfNet, then wrap
_base = KfNet(in_dim=IN_DIM, out_dim=M_DIM, hidden=256, n_blocks=5, dropout=0.15)
_base.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

model = KfNetFineTune(_base, hidden=256, out_dim=M_DIM, adapter_dim=128).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
frozen    = sum(p.numel() for p in model.backbone.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Loaded pretrained backbone  ({frozen:,} params frozen)')
print(f'Trainable ft_head params   : {trainable:,}')
print(f'Total params               : {total:,}')

## Generate fine-tuning data

In [ ]:
def grad_Jtilde(k, A_t, B_t, r):
    g = np.zeros(M_DIM, dtype=np.float64)
    k_sq = np.dot(k, k)
    for i in range(len(A_t)):
        d = A_t[i] + np.dot(B_t[i], k)
        if d >= -1e-14: continue
        num = np.dot(B_t[i], B_t[i]) + r * k_sq
        g  += -(r * k) / d + (num / (2.0 * d * d)) * B_t[i]
    return g

def make_feasible(k, A_t, B_t):
    k = k.copy()
    for _ in range(500):
        ok = True
        for i in range(len(A_t)):
            viol = A_t[i] + np.dot(B_t[i], k)
            if viol > -1e-9:
                ok = False
                nb_sq = np.dot(B_t[i], B_t[i])
                if nb_sq > 1e-14:
                    k -= (B_t[i] / nb_sq) * (viol + 1e-7)
        if ok: break
    return k

def _worker_compute(args):
    A_t, B_t, r_s = args
    return compute_k_star(A_t, B_t, r_s)

def compute_k_star(A_t, B_t, r_s):
    k0 = make_feasible(np.zeros(M_DIM), A_t, B_t)
    if not all(A_t[i] + np.dot(B_t[i], k0) < 0 for i in range(len(A_t))):
        return None
    def neg_grad(t, k): return -grad_Jtilde(k, A_t, B_t, r_s)
    def ev_conv(t, k):  return np.linalg.norm(grad_Jtilde(k, A_t, B_t, r_s)) - GRAD_TOL
    ev_conv.terminal = True; ev_conv.direction = -1
    try:
        sol = solve_ivp(neg_grad, [0.0, ODE_TMAX], k0, method='RK45',
                        rtol=ODE_RTOL, atol=ODE_ATOL,
                        events=ev_conv, dense_output=False, max_step=np.inf)
        ks = sol.y[:, -1]
        if np.linalg.norm(grad_Jtilde(ks, A_t, B_t, r_s)) > 1e-3: return None
        if not all(A_t[i] + np.dot(B_t[i], ks) < 0 for i in range(len(A_t))): return None
        return ks.astype(np.float32)
    except Exception: return None

print('Helpers defined.')

In [ ]:
def generate_finetune_data(n_samples, seed=42):
    pool = mp.Pool(1)
    rng = np.random.default_rng(seed)
    q_list, kstar_list = [], []
    accepted = attempted = 0
    t0 = time.time()

    while accepted < n_samples:
        attempted += 1

        n_cbf    = int(rng.integers(1, N_MAX))
        n_dim    = int(rng.integers(n_cbf, M_DIM + 1))
        N_active = 1 + n_cbf

        r_x = float(rng.uniform(X_RADIUS_MIN, X_RADIUS_MAX))
        x   = rng.standard_normal(M_DIM)
        x[:n_dim] = x[:n_dim] / (np.linalg.norm(x[:n_dim]) + 1e-15) * r_x
        x[n_dim:] = 0.0

        c_clf  = float(rng.uniform(C_CLF_MIN, C_CLF_MAX))
        A_list = [c_clf * float(np.dot(x, x))]
        B_list = [x.copy()]

        skip = False
        for _ in range(n_cbf):
            dist  = float(rng.uniform(OBS_DIST_MIN, OBS_DIST_MAX))
            c     = rng.standard_normal(M_DIM)
            c[:n_dim] = c[:n_dim] / (np.linalg.norm(c[:n_dim]) + 1e-15) * dist
            c[n_dim:] = 0.0
            xc    = x - c
            ds    = max(float(np.dot(xc, xc)), 1e-8)
            R     = float(rng.uniform(OBS_R_MIN, OBS_R_MAX))
            h     = CBF_GAIN * (1.0 - R*R / ds)
            if h <= 0.0: skip = True; break
            grad_h = CBF_GAIN * (2.0 * R*R / ds**2) * xc
            A_list.append(-h)
            B_list.append(-grad_h)
        if skip: continue

        M   = max(max(abs(a) for a in A_list),
                  max(np.linalg.norm(b) for b in B_list), 1.0)
        A_t = [a/M for a in A_list]
        B_t = [b/M for b in B_list]
        r_s = 1.0 / (M * M)

        result = pool.apply_async(_worker_compute, [(A_t, B_t, r_s)])
        try:
            k_star = result.get(timeout=20)
        except mp.TimeoutError:
            print("timeout — restarting worker")
            pool.terminate(); pool.join()
            pool = mp.Pool(1)
            k_star = None

        if k_star is None: continue

        q = np.zeros(IN_DIM, dtype=np.float32)
        for i in range(N_active):
            base = i * (M_DIM + 1)
            q[base]                = float(A_t[i])
            q[base+1:base+1+M_DIM] = B_t[i].astype(np.float32)
        a_dummy = float(-np.sqrt(r_s))
        for i in range(N_active, N_MAX):
            q[i * (M_DIM + 1)] = a_dummy
        q[-1] = float(r_s)

        q_list.append(q)
        kstar_list.append(k_star)
        accepted += 1

        if accepted % 50 == 0:
            elapsed = time.time() - t0
            rate    = accepted / elapsed
            eta     = (n_samples - accepted) / rate
            pct_acc = 100.0 * accepted / attempted
            print(f'  {accepted:>6}/{n_samples}  '
                  f'{rate:.0f} samples/s  '
                  f'accept={pct_acc:.0f}%  '
                  f'ETA={eta/60:.1f}min')

    elapsed   = time.time() - t0
    q_arr     = np.stack(q_list)
    kstar_arr = np.stack(kstar_list)
    print(f'Done: {n_samples} samples in {elapsed/60:.1f}min  ({n_samples/elapsed:.0f} samples/s)')
    print(f'  ||k*|| : mean={np.linalg.norm(kstar_arr,axis=1).mean():.3f}  '
          f'max={np.linalg.norm(kstar_arr,axis=1).max():.3f}')
    pool.close(); pool.join()
    return q_arr, kstar_arr, pool


print(f'Starting generation of {N_FINETUNE:,} samples...')
q_raw, kstar_raw, pool = generate_finetune_data(N_FINETUNE)
print(f'Generated: q={q_raw.shape}  k*={kstar_raw.shape}')

## Normalise with fixed scalers

In [ ]:
X_ft = scaler_X.transform(q_raw).astype(np.float32)
y_ft = scaler_y.transform(kstar_raw).astype(np.float32)

print(f'X_ft: mean={X_ft.mean():.3f}  std={X_ft.std():.3f}')
print(f'y_ft: mean={y_ft.mean():.3f}  std={y_ft.std():.3f}')
max_std = y_ft.std(axis=0).max()
print(f'y_ft max per-dim std: {max_std:.2f}  (>5 would indicate scaler mismatch)')

In [ ]:
n_val    = int(len(X_ft) * VAL_FRAC)
n_train  = len(X_ft) - n_val
dataset  = TensorDataset(torch.from_numpy(X_ft), torch.from_numpy(y_ft))
train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  pin_memory=False)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, pin_memory=False)
print(f'Train: {n_train:,}  Val: {n_val:,}')

## Fine-tune (frozen backbone, trainable ft_head only)

In [ ]:
# Only ft_head parameters are passed to the optimiser - backbone is frozen.
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=LR_FINETUNE / 10)
criterion = nn.HuberLoss(delta=1.0)

best_val  = float('inf')
patience  = 0
tr_losses, vl_losses = [], []
OUT_MODEL = os.path.join(WORK_DIR, 'best_kf_model_N10_m10_ft.pt')
print(f'Fine-tuning ft_head only  LR={LR_FINETUNE}  epochs={MAX_EPOCHS}  patience={PATIENCE}')

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    run = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.ft_head.parameters(), 1.0)
        optimizer.step()
        run += loss.item() * len(xb)
    tl = run / n_train
    scheduler.step()

    model.eval()
    run = 0.0
    with torch.no_grad():
        for xb, yb in val_dl:
            run += criterion(model(xb.to(DEVICE)), yb.to(DEVICE)).item() * len(xb)
    vl = run / n_val
    tr_losses.append(tl); vl_losses.append(vl)

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:>4} | train={tl:.5f} | val={vl:.5f} | lr={scheduler.get_last_lr()[0]:.2e}')

    if vl < best_val - 1e-7:
        best_val = vl; patience = 0
        torch.save(model.ft_head.state_dict(), OUT_MODEL)
    else:
        patience += 1
        if patience >= PATIENCE:
            print(f'Early stopping at epoch {epoch}.')
            break

print(f'Best val loss: {best_val:.6f}  ->  {OUT_MODEL}')

## Evaluate

In [ ]:
model.ft_head.load_state_dict(torch.load(OUT_MODEL, map_location=DEVICE))
model.eval()
preds, tgts = [], []
with torch.no_grad():
    for xb, yb in val_dl:
        preds.append(model(xb.to(DEVICE)).cpu().numpy())
        tgts.append(yb.numpy())
preds_orig = scaler_y.inverse_transform(np.vstack(preds))
tgts_orig  = scaler_y.inverse_transform(np.vstack(tgts))
print(f'Overall R2: {r2_score(tgts_orig, preds_orig, multioutput="uniform_average"):.4f}')
err = np.abs(preds_orig - tgts_orig)
print(f'MAE={err.mean():.4f}  P90={np.percentile(err,90):.4f}  P99={np.percentile(err,99):.4f}')

## Spot-check at simulation IC 1

In [ ]:
x_test  = np.array([4.0, 0.0, 1.0, -1.0, 2.0, 0.0, 1.0, -2.0, 0.5, -0.5])
obs_cs  = [np.array([2,0,0,0,0,2,0,0,0,0]), np.array([0,2,0,0,0,0,2,0,0,0]),
           np.array([0,0,2,0,0,0,0,2,0,0]), np.array([0,0,0,2,0,0,0,0,2,0]),
           np.array([0,0,0,0,2,0,0,0,0,2])]
obs_rs  = [0.8]*5

A_list = [0.5*float(np.dot(x_test,x_test))]; B_list = [x_test.copy()]
for c, R in zip(obs_cs, obs_rs):
    xc = x_test - c; ds = max(float(np.dot(xc,xc)), 1e-8)
    h = CBF_GAIN*(1 - R*R/ds); grad_h = CBF_GAIN*(2*R*R/ds**2)*xc
    A_list.append(-h); B_list.append(-grad_h)

M   = max(max(abs(a) for a in A_list), max(np.linalg.norm(b) for b in B_list), 1.0)
A_t = [a/M for a in A_list]; B_t = [b/M for b in B_list]; r_s = 1/(M*M)

q   = np.zeros(IN_DIM, dtype=np.float32)
for i in range(6):
    base = i*(M_DIM+1); q[base] = A_t[i]; q[base+1:base+1+M_DIM] = B_t[i]
for i in range(6, N_MAX):
    q[i*(M_DIM+1)] = float(-np.sqrt(r_s))
q[-1] = float(r_s)

q_sc = scaler_X.transform(q.reshape(1,-1)).astype(np.float32)
with torch.no_grad():
    z = model(torch.from_numpy(q_sc).to(DEVICE)).cpu().numpy()[0]
k_nn = scaler_y.inverse_transform(z.reshape(1,-1))[0]

# Compute k* for comparison
_pool = mp.Pool(1)
res = _pool.apply_async(_worker_compute, [(A_t, B_t, r_s)])
try:
    k_star = res.get(timeout=20)
except mp.TimeoutError:
    print("⚠️ timeout computing k*"); k_star = None
_pool.close(); _pool.join()

print(f'||k_nn||  = {np.linalg.norm(k_nn):.4f}')
if k_star is not None:
    err = np.linalg.norm(k_nn - k_star) / (np.linalg.norm(k_star)+1e-9)
    print(f'||k*||    = {np.linalg.norm(k_star):.4f}')
    print(f'Rel error = {err:.2%}')
A_mat = np.array(B_list); a_vec = np.array(A_list)
v = A_mat @ k_nn + a_vec
print(f'CLF constraint (k_nn): {v[0]:.4f}  (< 0 = satisfied)')

## Save outputs

In [ ]:
import shutil
for src, name in [(SCALER_X, 'scaler_X_N10_m10.pkl'), (SCALER_Y, 'scaler_y_N10_m10.pkl')]:
    dst = os.path.join(WORK_DIR, name)
    shutil.copy2(src, dst); print(f'Copied: {name}')

print()
print('=== Output files ===')
for f in ['best_kf_model_N10_m10_ft.pt', 'scaler_X_N10_m10.pkl',
          'scaler_y_N10_m10.pkl', 'learning_curves_finetune.png']:
    full = os.path.join(WORK_DIR, f)
    if os.path.exists(full):
        print(f'  OK      {f}  ({os.path.getsize(full)//1024} KB)')
    else:
        print(f'  MISSING {f}')